Классификация: CC50 > медианы выборки
Медиана CC50 ≈ 411.04 mM — бинарная классификация (0/1).

In [1]:
import os
os.makedirs("plots", exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, RocCurveDisplay, ConfusionMatrixDisplay
from sklearn.feature_selection import VarianceThreshold
import warnings

warnings.filterwarnings("ignore")

df = pd.read_excel("data/data.xlsx", index_col=0)
targets = ["IC50, mM", "CC50, mM", "SI"]
features = [c for c in df.columns if c not in targets]

X = df[features]
cc50_median = df["CC50, mM"].median()
y = (df["CC50, mM"] > cc50_median).astype(int)
print(f"Медиана CC50 = {cc50_median:.4f} mM")
print(f"Класс 1 (CC50 > медиана): {y.sum()} ({y.mean()*100:.1f}%)")

selector = VarianceThreshold(threshold=0.0)
X = X.fillna(X.median())
X_sel = selector.fit_transform(X)
sel_features = [features[i] for i in selector.get_support(indices=True)]
X = pd.DataFrame(X_sel, columns=sel_features)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Медиана CC50 = 411.0393 mM
Класс 1 (CC50 > медиана): 499 (49.9%)


## Базовые модели

In [2]:
print("\n── Базовые модели ──")
base_models = {
    "LogisticRegression": Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=1000))]),
    "SVM (RBF)":          Pipeline([("sc", StandardScaler()), ("m", SVC(probability=True))]),
    "RandomForest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
}
for name, model in base_models.items():
    acc = cross_val_score(model, X, y, cv=cv, scoring="accuracy").mean()
    auc = cross_val_score(model, X, y, cv=cv, scoring="roc_auc").mean()
    f1  = cross_val_score(model, X, y, cv=cv, scoring="f1").mean()
    print(f"  {name:22s}: Acc={acc:.3f}  AUC={auc:.3f}  F1={f1:.3f}")


── Базовые модели ──
  LogisticRegression    : Acc=0.738  AUC=0.815  F1=0.744
  SVM (RBF)             : Acc=0.748  AUC=0.831  F1=0.753
  RandomForest          : Acc=0.773  AUC=0.846  F1=0.773
  GradientBoosting      : Acc=0.755  AUC=0.845  F1=0.758


## GridSearchCV

In [3]:
print("\n── GridSearchCV: LogisticRegression ──")
gs_lr = GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=2000))]),
    {"m__C": [0.01, 0.1, 1, 10, 100]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_lr.fit(X, y)
print(f"  Best: {gs_lr.best_params_}  AUC={gs_lr.best_score_:.4f}")

print("\n── GridSearchCV: SVM ──")
gs_svm = GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("m", SVC(probability=True))]),
    {"m__C": [0.1, 1, 10], "m__gamma": ["scale", "auto"]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_svm.fit(X, y)
print(f"  Best: {gs_svm.best_params_}  AUC={gs_svm.best_score_:.4f}")

print("\n── GridSearchCV: RandomForest ──")
gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {"n_estimators": [100, 200], "max_depth": [None, 10, 20], "min_samples_split": [2, 5]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_rf.fit(X, y)
print(f"  Best: {gs_rf.best_params_}  AUC={gs_rf.best_score_:.4f}")

print("\n── GridSearchCV: GradientBoosting ──")
gs_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1, 0.2], "max_depth": [3, 5]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_gb.fit(X, y)
print(f"  Best: {gs_gb.best_params_}  AUC={gs_gb.best_score_:.4f}")


── GridSearchCV: LogisticRegression ──
  Best: {'m__C': 0.1}  AUC=0.8251

── GridSearchCV: SVM ──
  Best: {'m__C': 1, 'm__gamma': 'auto'}  AUC=0.8312

── GridSearchCV: RandomForest ──
  Best: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}  AUC=0.8540

── GridSearchCV: GradientBoosting ──
  Best: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100}  AUC=0.8455


## Итог

In [4]:
print("\n── Итоговое сравнение ROC-AUC (tuned) ──")
tuned_auc = {
    "LogisticRegression": gs_lr.best_score_,
    "SVM":                gs_svm.best_score_,
    "RandomForest":       gs_rf.best_score_,
    "GradientBoosting":   gs_gb.best_score_,
}
for name, auc in sorted(tuned_auc.items(), key=lambda x: -x[1]):
    print(f"  {name:22s}: AUC = {auc:.4f}")


── Итоговое сравнение ROC-AUC (tuned) ──
  RandomForest          : AUC = 0.8540
  GradientBoosting      : AUC = 0.8455
  SVM                   : AUC = 0.8312
  LogisticRegression    : AUC = 0.8251


## ROC-кривые

In [5]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, gs in [("LogReg", gs_lr), ("SVM", gs_svm), ("RF", gs_rf), ("GB", gs_gb)]:
    RocCurveDisplay.from_estimator(gs.best_estimator_, X, y, ax=ax, name=name)
ax.set_title("ROC-кривые (CC50 > медианы)")
plt.tight_layout()
plt.savefig("plots/cls_cc50_roc.png")
plt.close()
print("\nСохранён график: cls_cc50_roc.png")


Сохранён график: cls_cc50_roc.png


## Матрица ошибок

In [6]:
best_gs = max([gs_lr, gs_svm, gs_rf, gs_gb], key=lambda g: g.best_score_)
y_pred = best_gs.predict(X)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y, y_pred, ax=ax, colorbar=False)
ax.set_title("Матрица ошибок (лучшая модель, CC50 > медианы)")
plt.tight_layout()
plt.savefig("plots/cls_cc50_cm.png")
plt.close()
print("\nClassification Report:")
print(classification_report(y, y_pred, target_names=["CC50 ≤ медианы", "CC50 > медианы"]))



Classification Report:
                precision    recall  f1-score   support

CC50 ≤ медианы       0.95      0.96      0.96       502
CC50 > медианы       0.96      0.95      0.96       499

      accuracy                           0.96      1001
     macro avg       0.96      0.96      0.96      1001
  weighted avg       0.96      0.96      0.96      1001



## Итоговые результаты

- Задача бинарной классификации: предсказать, является ли соединение малотоксичным (CC50 > 411 мМ) или высокотоксичным (CC50 ≤ медианы). Классы сбалансированы – 49.9% и 50.1%.
- Базовые модели показывают хорошее качество. Лучшая базовая модель – RandomForest с Accuracy=0.773, AUC=0.846, F1=0.773.
- Настройка гиперпараметров улучшает все модели. Лучший результат у RandomForest с параметрами (max_depth=10, min_samples_split=2, n_estimators=100) – AUC=0.8540.
- GradientBoosting после настройки дает близкий результат – AUC=0.8455. SVM и логистическая регрессия немного отстают (AUC=0.831 и 0.825 соответственно).
- Качество предсказания токсичности (AUC=0.854) заметно выше, чем для активности IC50 (AUC=0.795). Это означает, что молекулярные дескрипторы лучше предсказывают токсичность, чем эффективность.
- Разрыв между линейной моделью (AUC=0.825) и лучшей древовидной (AUC=0.854) составляет 0.029 – нелинейность присутствует, но не критична.
- Итоговое качество (AUC≈0.85) можно считать хорошим. Модель способна правильно различить малотоксичные и высокотоксичные соединения в 85% случаев.